In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ATHLETIQ = Path("/content/drive/MyDrive/AthletiQ")

print("AthletiQ path:", DRIVE_ATHLETIQ)
print("Exists:", DRIVE_ATHLETIQ.exists())

if DRIVE_ATHLETIQ.exists():
    print("\nContents:")
    for p in DRIVE_ATHLETIQ.iterdir():
        print(" ", p.name)

Mounted at /content/drive
AthletiQ path: /content/drive/MyDrive/AthletiQ
Exists: True

Contents:
  soccernet_ball_yolo.zip
  models


In [2]:
import shutil
from pathlib import Path

DRIVE_ZIP = DRIVE_ATHLETIQ / "soccernet_ball_yolo.zip"
RUNTIME_ZIP = Path("/content/soccernet_ball_yolo.zip")

shutil.copy2(DRIVE_ZIP, RUNTIME_ZIP)

print("✓ ZIP copied to runtime")
print("Path:", RUNTIME_ZIP)
print("Size:", f"{RUNTIME_ZIP.stat().st_size / (1024**3):.2f} GB")

✓ ZIP copied to runtime
Path: /content/soccernet_ball_yolo.zip
Size: 17.05 GB


In [3]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used:  {used / (1024**3):.2f} GB")
print(f"Free:  {free / (1024**3):.2f} GB")

Total: 112.64 GB
Used:  59.58 GB
Free:  53.05 GB


In [4]:
import zipfile

RUNTIME_ZIP = Path("/content/soccernet_ball_yolo.zip")

with zipfile.ZipFile(RUNTIME_ZIP, "r") as z:
    total_uncompressed = sum(info.file_size for info in z.infolist())
    file_count = len(z.infolist())

print(f"Files in ZIP: {file_count:,}")
print(f"Uncompressed size: {total_uncompressed / (1024**3):.2f} GB")

Files in ZIP: 159,011
Uncompressed size: 17.05 GB


In [5]:
import zipfile
from pathlib import Path

RUNTIME_ZIP = Path("/content/soccernet_ball_yolo.zip")
EXTRACT_DIR = Path("/content/soccernet_ball_yolo")

EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(RUNTIME_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

print("✓ Extraction complete")
print("Dataset path:", EXTRACT_DIR)

print("\nTop-level contents:")
for p in EXTRACT_DIR.iterdir():
    print(" ", p.name)

✓ Extraction complete
Dataset path: /content/soccernet_ball_yolo

Top-level contents:
  soccernet_ball_yolo


In [6]:
from pathlib import Path

DATASET_ROOT = Path("/content/soccernet_ball_yolo/soccernet_ball_yolo")

for split in ["train", "val", "test"]:
    images = list((DATASET_ROOT / "images" / split).glob("*"))
    labels = list((DATASET_ROOT / "labels" / split).glob("*.txt"))

    print(
        f"{split:5} | "
        f"images: {len(images):,} | "
        f"labels: {len(labels):,}"
    )

print("\nDataset root:", DATASET_ROOT)

train | images: 55,500 | labels: 55,500
val   | images: 11,250 | labels: 11,250
test  | images: 12,750 | labels: 12,750

Dataset root: /content/soccernet_ball_yolo/soccernet_ball_yolo


In [7]:
import shutil
from pathlib import Path

MODEL_DIR = DRIVE_ATHLETIQ / "models"

print("Models in Drive:")
for p in MODEL_DIR.iterdir():
    print(" ", p.name)

Models in Drive:
  football_ball_detector.pt


In [8]:
MODEL_SRC = DRIVE_ATHLETIQ / "models" / "football_ball_detector.pt"
MODEL_PATH = Path("/content/football_ball_detector.pt")

shutil.copy2(MODEL_SRC, MODEL_PATH)

print("✓ Model copied to runtime")
print("Path:", MODEL_PATH)
print("Size:", f"{MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

✓ Model copied to runtime
Path: /content/football_ball_detector.pt
Size: 48.89 MB


In [10]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 8.7 MB/s eta 0:00:00


In [11]:
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ball_model = YOLO(str(MODEL_PATH))

print("\n✓ Ball detector loaded")
print("Model:", MODEL_PATH.name)
print("Classes:", ball_model.names)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

✓ Ball detector loaded
Model: football_ball_detector.pt
Classes: {0: 'ball'}


In [14]:
DATA_YAML = Path("/content/soccernet_ball.yaml")

DATA_YAML.write_text(f"""
path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: ball
""")

print(DATA_YAML.read_text())


path: /content/soccernet_ball_yolo/soccernet_ball_yolo
train: images/train
val: images/val
test: images/test

names:
  0: ball



In [15]:
val_results = ball_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    conf=0.001,
    verbose=False
)

print("Validation")
print("Precision :", val_results.box.mp)
print("Recall    :", val_results.box.mr)
print("mAP50     :", val_results.box.map50)
print("mAP50-95  :", val_results.box.map)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11l summary (fused): 190 layers, 25,280,083 parameters, 0 gradients, 86.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 898.9±1036.3 MB/s, size: 233.2 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/val.cache... 11250 images, 986 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11250/11250 1.9Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 352/352 1.3it/s 4:31
                   all      11250      10636      0.532        0.3      0.263     0.0863
Speed: 0.8ms preprocess, 18.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/detect/val-3
Validation
Precision : 0.5315294133060435
Recall    : 0.30007953783601676
mAP50     : 0.26285789749985766
mAP50-95  : 0.08626111153157827


In [16]:
test_results = ball_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=32,
    device=0,
    conf=0.001,
    verbose=False
)

print("Test")
print("Precision :", test_results.box.mp)
print("Recall    :", test_results.box.mr)
print("mAP50     :", test_results.box.map50)
print("mAP50-95  :", test_results.box.map)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11l summary (fused): 190 layers, 25,280,083 parameters, 0 gradients, 86.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 59.3±31.4 MB/s, size: 206.0 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test... 12750 images, 1240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12750/12750 501.4it/s 25.4s
val: New cache created: /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 399/399 1.3it/s 5:05
                   all      12750      11844      0.547      0.311      0.282     0.0964
Speed: 0.7ms preprocess, 19.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val-4
Test
Precision : 0.5472126947187296
Recall    : 0.31070584262073625
mAP50     : 0.2824873330622961
mAP50-95  : 0.09641770267411881


In [17]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 11778 MiB


In [20]:
train_results = ball_model.train(
    data=str(DATA_YAML),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=8,
    project="/content/ball_training",
    name="yolo11l_soccernet",
    exist_ok=True,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/soccernet_ball.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/football_ball_detector.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11l_soccernet, nbs=64, nms=None, o

In [21]:
from ultralytics import YOLO
from pathlib import Path

# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

BEST_PT = "/content/ball_training/yolo11l_soccernet/weights/best.pt"
DATA_YAML = "/content/soccernet_ball.yaml"

model = YOLO(BEST_PT)

# ============================================================
# TEST SET ONLY
# ============================================================

test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    verbose=True
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("YOLO11L SOCCERNET — 5 EPOCH TEST RESULTS")
print("=" * 70)

print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP50     : {test_results.box.map50:.4f}")
print(f"mAP50-95  : {test_results.box.map:.4f}")
print("=" * 70)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11l summary (fused): 190 layers, 25,280,083 parameters, 0 gradients, 86.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 90.3±29.8 MB/s, size: 246.7 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test.cache... 12750 images, 1240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12750/12750 3.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 797/797 2.4it/s 5:32
                   all      12750      11844      0.684      0.502      0.497      0.184
Speed: 0.7ms preprocess, 20.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val-5

YOLO11L SOCCERNET — 5 EPOCH TEST RESULTS
Precision : 0.6836
Recall    : 0.5024
mAP50     : 0.4973
mAP50-95  : 0.1836
